# 02b Weak-Modality Robustness Probe

Objective: test whether the pilot `nice_to_have` to `recommended` collapse is tied to one wording or generalizes across weak stakeholder-intent phrasings.

This formative probe uses the same 20 pilot seeds, Task 2 only, and the existing modality extraction prompt. It does not change the main benchmark.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
DATASET_ID = eu.normalize_dataset_id(os.getenv("DATASET_ID", "nice"))
DATASET_SUFFIX = eu.dataset_suffix(DATASET_ID)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)
ARTIFACT_SUFFIX = eu.dataset_variant_suffix(DATASET_ID, BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, DATASET_ID, BENCHMARK_VARIANT


## Configure Probe


In [ ]:
HOST = os.getenv("HOST", CONFIG["llm"]["host"])
configured_models = [m.strip() for m in os.getenv("MODELS", ",".join(CONFIG["llm"]["models"])).split(",") if m.strip()]
MODEL = os.getenv("MODEL", configured_models[0])
RUN_WEAK_MODALITY_PROBE = os.getenv("RUN_WEAK_MODALITY_PROBE", "true").lower() in {"1", "true", "yes"}
RUN_WEAK_MODALITY_STOCHASTIC = os.getenv("RUN_WEAK_MODALITY_STOCHASTIC", "false").lower() in {"1", "true", "yes"}
REQUEST_CONCURRENCY = eu.resolve_llm_concurrency(CONFIG)

deterministic = CONFIG["llm"]["deterministic"]
stochastic = CONFIG["llm"]["stochastic"]
print({
    "HOST": HOST,
    "MODEL": MODEL,
    "RUN_WEAK_MODALITY_PROBE": RUN_WEAK_MODALITY_PROBE,
    "RUN_WEAK_MODALITY_STOCHASTIC": RUN_WEAK_MODALITY_STOCHASTIC,
    "REQUEST_CONCURRENCY": REQUEST_CONCURRENCY,
    "BENCHMARK_VARIANT": BENCHMARK_VARIANT,
})


## Build Probe Items


In [ ]:
benchmark_path = eu.artifact_path(PROJECT_ROOT / "data/processed/benchmark_items.csv", DATASET_ID, BENCHMARK_VARIANT)
seeds_path = eu.artifact_path(PROJECT_ROOT / "data/processed/seeds_selected.csv", DATASET_ID)
probe_items_path = eu.artifact_path(PROJECT_ROOT / "data/processed/weak_modality_probe_items.csv", DATASET_ID, BENCHMARK_VARIANT)

benchmark = eu.read_csv_rows(benchmark_path)
seed_rows = eu.read_csv_rows(seeds_path)
pilot_seed_count = int(CONFIG["project"]["pilot_seed_count"])
pilot_seed_ids = sorted({row["seed_id"] for row in benchmark})[:pilot_seed_count]
pilot_seed_order = {seed_id: index for index, seed_id in enumerate(pilot_seed_ids)}
pilot_seeds = sorted(
    [row for row in seed_rows if row["seed_id"] in pilot_seed_order],
    key=lambda row: pilot_seed_order[row["seed_id"]],
)
assert len(pilot_seeds) == pilot_seed_count

probe_items = eu.build_weak_modality_probe_items(pilot_seeds)
expected_items = pilot_seed_count * len(eu.WEAK_MODALITY_PROBE_TEMPLATES)
assert len(probe_items) == expected_items
assert len({row["item_id"] for row in probe_items}) == expected_items
assert {row["task2_gold_modality"] for row in probe_items} == {"nice_to_have"}

eu.write_csv_rows(probe_items_path, probe_items, fieldnames=eu.WEAK_MODALITY_PROBE_FIELDS)
print(f"Probe items: {len(probe_items)}")
print(f"Wrote probe items: {probe_items_path}")
print(eu.markdown_table(probe_items[:8], ["item_id", "template_id", "source_statement", "task2_gold_modality"]))


## Pre-Model Sanity Check


In [ ]:
sanity_paths = eu.write_weak_modality_template_sanity_check(PROJECT_ROOT / "outputs", suffix=DATASET_SUFFIX)
sanity_rows = eu.read_csv_rows(sanity_paths["csv"])
sanity_status = eu.weak_modality_sanity_status(sanity_rows)
PROBE_READY = bool(sanity_status["valid"])

print(f"Sanity CSV: {sanity_paths['csv']}")
print(f"Sanity Markdown: {sanity_paths['markdown']}")
print(sanity_status)
if not PROBE_READY:
    print("Probe is gated: mark each template as weaker_than_should=yes in the sanity CSV before model execution.")
print(eu.markdown_table(sanity_rows, eu.WEAK_MODALITY_SANITY_FIELDS))


## Run Task 2 Probe


In [ ]:
task1_template = eu.load_prompt(PROJECT_ROOT / "prompts/mandatory_entailment.txt")
task2_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_extraction.txt")

def prompt_for(task, item):
    if task == "task1":
        return eu.render_prompt(
            task1_template,
            source_statement=item["source_statement"],
            candidate_requirement=item["candidate_requirement"],
        )
    if task == "task2":
        return eu.render_prompt(task2_template, source_statement=item["source_statement"])
    raise ValueError(task)

def request_job(
    item,
    task,
    model,
    sample_kind,
    sample_index,
    temperature,
    top_p,
    run_id,
    request_index,
    prompt=None,
    prompt_version=None,
):
    prompt = prompt if prompt is not None else prompt_for(task, item)
    return {
        "request_index": request_index,
        "run_id": run_id,
        "model": model,
        "host": HOST,
        "task": task,
        "item": item,
        "sample_index": sample_index,
        "sample_kind": sample_kind,
        "temperature": temperature,
        "top_p": top_p,
        "prompt_version": prompt_version or CONFIG["project"]["prompt_version"],
        "prompt": prompt,
        "max_tokens": int(CONFIG["llm"]["max_tokens"]),
        "timeout_s": int(CONFIG["llm"]["timeout_s"]),
        "api_key_env": CONFIG["llm"]["api_key_env"],
    }


In [ ]:
output_path = eu.artifact_path(PROJECT_ROOT / "data/processed/model_outputs_raw_weak_modality_probe.jsonl", DATASET_ID, BENCHMARK_VARIANT)
run_id = eu.new_run_id("weak-modality-probe" if BENCHMARK_VARIANT == "must" else f"weak-modality-probe-{BENCHMARK_VARIANT}")
records = []
jobs = []

for item in probe_items:
    jobs.append(request_job(
        item=item,
        task="task2",
        model=MODEL,
        sample_kind="deterministic",
        sample_index=0,
        temperature=float(deterministic["temperature"]),
        top_p=float(deterministic["top_p"]),
        run_id=run_id,
        request_index=len(jobs),
    ))
    if RUN_WEAK_MODALITY_STOCHASTIC:
        for sample_index in range(int(stochastic["samples"])):
            jobs.append(request_job(
                item=item,
                task="task2",
                model=MODEL,
                sample_kind="stochastic",
                sample_index=sample_index,
                temperature=float(stochastic["temperature"]),
                top_p=float(stochastic["top_p"]),
                run_id=run_id,
                request_index=len(jobs),
            ))

if RUN_WEAK_MODALITY_PROBE and PROBE_READY:
    print(f"Dispatching {len(jobs)} weak-modality probe calls with concurrency={REQUEST_CONCURRENCY}")
    for record in eu.run_completion_jobs(jobs, max_workers=REQUEST_CONCURRENCY):
        eu.append_jsonl(output_path, record)
        records.append(record)
        if len(records) % 20 == 0 or len(records) == len(jobs):
            print(f"Completed {len(records)}/{len(jobs)} weak-modality probe calls")
    print(f"Wrote {len(records)} weak-modality probe records to {output_path}")
elif not PROBE_READY:
    print("Weak-modality probe not run because the sanity check is incomplete.")
else:
    print("Weak-modality probe not run. Set RUN_WEAK_MODALITY_PROBE=true or edit the flag to True.")


## Summarize Probe


In [ ]:
if records:
    probe_run_id, probe_rows = run_id, records
elif output_path.exists():
    probe_run_id, probe_rows = eu.select_run_rows(
        eu.read_jsonl(output_path),
        prefix="weak-modality-probe" if BENCHMARK_VARIANT == "must" else f"weak-modality-probe-{BENCHMARK_VARIANT}",
    )
else:
    probe_run_id, probe_rows = None, []

summary = eu.weak_modality_probe_summary(probe_items, probe_rows)
summary_paths = eu.write_weak_modality_probe_summary(summary, PROJECT_ROOT / "outputs", suffix=DATASET_SUFFIX)
print(f"Selected probe run: {probe_run_id}")
print(f"Wrote summary CSV: {summary_paths['csv']}")
print(f"Wrote summary Markdown: {summary_paths['markdown']}")
print(eu.markdown_table(summary, eu.WEAK_MODALITY_PROBE_SUMMARY_FIELDS))


## Decision Rule


In [ ]:
print("Interpretation guide:")
print("- Most templates collapse to recommended: proceed to full runs with robustness note.")
print("- Only useful_if collapses: treat the pilot as phrase-specific and revise or narrow the claim.")
print("- Mixed labels: proceed cautiously and frame weak modality as lexically sensitive.")
print("- Sanity check not valid: hold and revise the weak-modality construct.")
